# 🎬 Lab Setup: Content Recommendation Engine

This notebook generates **synthetic streaming platform data** for the Content Recommendation Engine lab.

**What gets created:**
- `main.content_reco_demo.users` — 500 subscriber profiles with preferences
- `main.content_reco_demo.content_catalog` — 200 movies/shows with metadata
- `main.content_reco_demo.viewing_history` — ~15,000 viewing events with ratings

**Prerequisites:** Unity Catalog access with permissions to create schemas in the `main` catalog.

> ⏱️ **Estimated runtime:** ~2 minutes on serverless compute

In [0]:
%sql
-- Create a dedicated schema for this lab
CREATE SCHEMA IF NOT EXISTS main.content_reco_demo
COMMENT 'Content Recommendation Engine demo schema for Lab 08';

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import random
import uuid
from datetime import datetime, timedelta

random.seed(42)

# --- User Generation ---
SCHEMA = "main.content_reco_demo"
NUM_USERS = 500

subscription_tiers = ["free", "basic", "standard", "premium"]
tier_weights = [0.15, 0.30, 0.35, 0.20]

genres = ["Action", "Comedy", "Drama", "Sci-Fi", "Horror", "Romance", "Documentary", "Thriller", "Animation", "Fantasy"]
regions = ["US-East", "US-West", "US-Central", "EU-West", "EU-East", "APAC", "LATAM"]

users = []
for i in range(1, NUM_USERS + 1):
    signup_date = datetime(2023, 1, 1) + timedelta(days=random.randint(0, 900))
    # Each user prefers 2-4 genres
    num_preferred = random.randint(2, 4)
    preferred_genres = random.sample(genres, num_preferred)
    tier = random.choices(subscription_tiers, weights=tier_weights, k=1)[0]
    
    users.append((
        i,
        f"user_{i:04d}",
        signup_date.strftime("%Y-%m-%d"),
        tier,
        "|".join(preferred_genres),
        random.choice(regions),
        random.randint(18, 70)
    ))

users_schema = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("username", StringType(), False),
    StructField("signup_date", StringType(), False),
    StructField("subscription_tier", StringType(), False),
    StructField("preferred_genres", StringType(), False),
    StructField("region", StringType(), False),
    StructField("age", IntegerType(), False)
])

df_users = spark.createDataFrame(users, schema=users_schema) \
    .withColumn("signup_date", F.to_date("signup_date")) \
    .withColumn("user_id", F.col("user_id").cast("int"))

df_users.write.mode("overwrite").saveAsTable(f"{SCHEMA}.users")
print(f"✅ Created {SCHEMA}.users with {NUM_USERS} records")
display(df_users.limit(5))

In [0]:
# --- Content Catalog Generation ---
NUM_CONTENT = 200

content_types = ["movie", "series", "documentary", "special"]
content_type_weights = [0.45, 0.35, 0.12, 0.08]

movie_prefixes = ["The", "A", "Dark", "Last", "Secret", "Final", "Lost", "Eternal", "Silent", "Golden"]
movie_nouns = ["Journey", "Kingdom", "Legacy", "Shadow", "Storm", "Phoenix", "Horizon", "Cipher", "Protocol", "Paradox",
               "Frontier", "Catalyst", "Mirage", "Eclipse", "Heist", "Requiem", "Uprising", "Covenant", "Reckoning", "Labyrinth"]

content_items = []
for i in range(1, NUM_CONTENT + 1):
    title = f"{random.choice(movie_prefixes)} {random.choice(movie_nouns)} {random.choice(['', 'II', 'III', 'Returns', 'Reloaded', 'Unleashed'])}".strip()
    genre = random.choice(genres)
    content_type = random.choices(content_types, weights=content_type_weights, k=1)[0]
    release_year = random.randint(2018, 2026)
    
    # Duration depends on type
    if content_type == "movie":
        duration = random.randint(85, 180)
    elif content_type == "series":
        duration = random.randint(25, 60)  # per episode
    elif content_type == "documentary":
        duration = random.randint(45, 120)
    else:
        duration = random.randint(30, 90)
    
    content_items.append((
        i,
        f"{title} ({i})",  # ensure uniqueness
        genre,
        content_type,
        release_year,
        duration,
        round(random.uniform(5.0, 9.5), 1),  # critic_score
        random.choice(["English", "Spanish", "French", "Korean", "Japanese", "German"])
    ))

content_schema = StructType([
    StructField("content_id", IntegerType(), False),
    StructField("title", StringType(), False),
    StructField("genre", StringType(), False),
    StructField("content_type", StringType(), False),
    StructField("release_year", IntegerType(), False),
    StructField("duration_minutes", IntegerType(), False),
    StructField("critic_score", FloatType(), True),
    StructField("language", StringType(), False)
])

df_content = spark.createDataFrame(content_items, schema=content_schema)
df_content.write.mode("overwrite").saveAsTable(f"{SCHEMA}.content_catalog")
print(f"✅ Created {SCHEMA}.content_catalog with {NUM_CONTENT} records")
display(df_content.limit(5))

In [0]:
# --- Viewing History Generation ---
# Simulate realistic viewing patterns:
# - Users watch more content in their preferred genres
# - Premium users watch more overall
# - Ratings correlate with genre preference match

import itertools

NUM_VIEWS = 15000

# Load user preferences for realistic simulation
user_prefs = {row.user_id: row.preferred_genres.split("|") for row in df_users.collect()}
user_tiers = {row.user_id: row.subscription_tier for row in df_users.collect()}
content_genres = {row.content_id: row.genre for row in df_content.collect()}

# Tier -> average number of views
tier_view_multiplier = {"free": 0.6, "basic": 0.8, "standard": 1.0, "premium": 1.4}
device_types = ["smart_tv", "mobile", "tablet", "laptop", "gaming_console"]
device_weights = [0.35, 0.25, 0.15, 0.20, 0.05]

viewing_records = []
view_id = 0

for _ in range(NUM_VIEWS):
    user_id = random.randint(1, NUM_USERS)
    content_id = random.randint(1, NUM_CONTENT)
    
    # Bias toward preferred genres (70% chance of picking preferred genre content)
    if random.random() < 0.7:
        preferred = user_prefs.get(user_id, genres[:2])
        # Find content matching preferred genres
        matching = [cid for cid, g in content_genres.items() if g in preferred]
        if matching:
            content_id = random.choice(matching)
    
    genre_match = content_genres.get(content_id, "") in user_prefs.get(user_id, [])
    
    # Viewing timestamp: spread across 2024-2026
    view_ts = datetime(2024, 1, 1) + timedelta(
        days=random.randint(0, 800),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59)
    )
    
    # Watch completion % - higher for genre matches
    if genre_match:
        watch_pct = min(100, max(10, random.gauss(78, 20)))
    else:
        watch_pct = min(100, max(5, random.gauss(45, 25)))
    
    # Rating (1-5) - correlated with watch completion and genre match
    if watch_pct > 70:
        rating = min(5.0, max(1.0, round(random.gauss(4.0, 0.7), 1)))
    elif watch_pct > 40:
        rating = min(5.0, max(1.0, round(random.gauss(3.0, 0.8), 1)))
    else:
        rating = min(5.0, max(1.0, round(random.gauss(2.0, 0.9), 1)))
    
    # ~30% of views don't have a rating
    if random.random() < 0.30:
        rating = None
    
    view_id += 1
    viewing_records.append((
        view_id,
        user_id,
        content_id,
        view_ts.strftime("%Y-%m-%d %H:%M:%S"),
        round(watch_pct, 1),
        rating,
        random.choices(device_types, weights=device_weights, k=1)[0]
    ))

viewing_schema = StructType([
    StructField("view_id", IntegerType(), False),
    StructField("user_id", IntegerType(), False),
    StructField("content_id", IntegerType(), False),
    StructField("view_timestamp", StringType(), False),
    StructField("watch_pct", FloatType(), True),
    StructField("rating", FloatType(), True),
    StructField("device_type", StringType(), False)
])

df_views = spark.createDataFrame(viewing_records, schema=viewing_schema) \
    .withColumn("view_timestamp", F.to_timestamp("view_timestamp"))

df_views.write.mode("overwrite").saveAsTable(f"{SCHEMA}.viewing_history")
print(f"✅ Created {SCHEMA}.viewing_history with {view_id} records")
display(df_views.limit(5))

In [0]:
# --- Verification ---
print("=" * 60)
print("📊 DATA GENERATION SUMMARY")
print("=" * 60)

tables = ["users", "content_catalog", "viewing_history"]
for table in tables:
    count = spark.table(f"{SCHEMA}.{table}").count()
    print(f"  ✅ {SCHEMA}.{table}: {count:,} rows")

print("\n" + "=" * 60)
print("🎯 Setup complete! Proceed to the main lab notebook.")
print("=" * 60)

## ✅ Setup Complete!

The following tables are now available in Unity Catalog:

| Table | Description | Rows |
|-------|-------------|------|
| `main.content_reco_demo.users` | Subscriber profiles with genre preferences | 500 |
| `main.content_reco_demo.content_catalog` | Movies, series, and documentaries | 200 |
| `main.content_reco_demo.viewing_history` | Timestamped viewing events with ratings | ~15,000 |

### Data Design Notes
- **Genre preference bias**: 70% of views align with user preferences (realistic signal for collaborative filtering)
- **Rating sparsity**: ~30% of views have no rating (mimics real-world optional ratings)
- **Temporal spread**: Views span 2024–2026 for point-in-time feature engineering
- **Tier stratification**: Premium users have higher engagement rates

**Next →** Open `01_Content_Recommendation_Engine` to begin the lab.